# MeterMind — Milestone 2: Deterministic Reconciliation Engine

**Goal:** detect billing anomalies using deterministic Python before adding LLM agents.

This notebook loads the synthetic fixture exported by Milestone 1, runs reconciliation checks, and produces structured findings for later LangChain tools and LangGraph agents.

**Important:** run this notebook from `C:\Projects\MeterMind` (or change `FIXTURE_PATH` below). It does not modify real invoices or connect to a payment provider.


In [ ]:
from pathlib import Path
import json
from collections import Counter, defaultdict
from decimal import Decimal, ROUND_HALF_UP
from typing import Any

FIXTURE_PATH = Path("metermind_synthetic_fixture.json")

if not FIXTURE_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {FIXTURE_PATH.resolve()}. "
        "Run this notebook from C:\\Projects\\MeterMind or set FIXTURE_PATH to the full path."
    )

with FIXTURE_PATH.open("r", encoding="utf-8") as f:
    fixture = json.load(f)

print("Loaded fixture:", FIXTURE_PATH.resolve())
print("Top-level keys:", list(fixture.keys()))


## 1. Inspect the fixture schema

The following cell prints record counts and a sample of each dataset. If your Milestone 1 fixture uses slightly different key names, use the output to update the small `get_records()` adapter below rather than silently guessing.


In [ ]:
def get_records(*candidate_keys: str) -> list[dict[str, Any]]:
    for key in candidate_keys:
        value = fixture.get(key)
        if isinstance(value, list):
            return value
    raise KeyError(
        f"None of these keys exist as lists: {candidate_keys}. "
        f"Available keys: {list(fixture.keys())}"
    )

# Candidate aliases make the loader tolerant of small naming differences.
clean_events = get_records("clean_events", "source_events", "original_events")
observed_events = get_records("observed_events", "processed_events", "ingested_events")
baseline_invoice_lines = get_records("baseline_invoice_lines", "correct_invoice_lines", "expected_invoice_lines")
observed_invoice_lines = get_records("observed_invoice_lines", "actual_invoice_lines", "invoice_lines")

print(f"Clean/source events: {len(clean_events)}")
print(f"Observed events: {len(observed_events)}")
print(f"Baseline invoice lines: {len(baseline_invoice_lines)}")
print(f"Observed invoice lines: {len(observed_invoice_lines)}")
print("\nSample clean event:")
print(json.dumps(clean_events[0], indent=2))
print("\nSample baseline invoice line:")
print(json.dumps(baseline_invoice_lines[0], indent=2))


## 2. Normalize fields and money

The fixture serializes decimal values as strings so financial values are not silently converted to binary floating-point numbers. We normalize amounts to `Decimal` and use explicit field adapters.


In [ ]:
def first_present(record: dict[str, Any], *keys: str, default=None):
    for key in keys:
        if key in record:
            return record[key]
    return default

def event_id(event: dict[str, Any]) -> str:
    value = first_present(event, "event_id", "id", "usage_event_id")
    if value is None:
        raise KeyError(f"Event has no recognized ID field: {event}")
    return str(value)

def invoice_line_id(line: dict[str, Any]) -> str:
    value = first_present(line, "invoice_line_id", "line_id", "id")
    if value is None:
        raise KeyError(f"Invoice line has no recognized ID field: {line}")
    return str(value)

def decimal_value(value: Any) -> Decimal:
    if value is None:
        return Decimal("0")
    return Decimal(str(value))

def money(value: Any) -> Decimal:
    return decimal_value(value).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

def line_amount(line: dict[str, Any]) -> Decimal:
    value = first_present(line, "amount", "line_amount", "total")
    if value is None:
        quantity = decimal_value(first_present(line, "quantity", default=0))
        unit_price = decimal_value(first_present(line, "unit_price", "price_per_unit", default=0))
        value = quantity * unit_price
    return money(value)

def event_signature(event: dict[str, Any]) -> tuple:
    # Event IDs are the primary identity. The signature is useful for audit evidence.
    return (
        event_id(event),
        str(first_present(event, "customer_id", default="")),
        str(first_present(event, "occurred_at", "timestamp", default="")),
        str(first_present(event, "quantity", default="")),
        str(first_present(event, "metric", default="")),
    )

print("Normalization helpers ready.")


## 3. Detect duplicate events

A duplicate is detected when the same event ID appears more than once in the observed stream. We return structured findings instead of just printing a message so later agents can consume the output.


In [ ]:
def detect_duplicate_events(events: list[dict[str, Any]]) -> list[dict[str, Any]]:
    counts = Counter(event_id(event) for event in events)
    findings = []
    for duplicate_id, count in sorted(counts.items()):
        if count > 1:
            matching = [event for event in events if event_id(event) == duplicate_id]
            findings.append({
                "finding_type": "duplicate_event",
                "severity": "high",
                "event_id": duplicate_id,
                "occurrences": count,
                "duplicate_copies": count - 1,
                "customer_ids": sorted({
                    str(first_present(event, "customer_id", default="unknown"))
                    for event in matching
                }),
                "evidence": {"event_signatures": [event_signature(event) for event in matching]},
                "requires_review": True,
            })
    return findings

duplicate_findings = detect_duplicate_events(observed_events)
print(json.dumps(duplicate_findings, indent=2, default=str))


## 4. Detect missing events

A missing event can only be proven when we have an authoritative source ledger to compare against. In this synthetic fixture, `clean_events` is that source ledger and `observed_events` is the downstream stream.

In a real integration, MeterMind would need an upstream provider ledger, webhook archive, or other independent source of expected events. Without one, it should report an unexplained usage gap rather than claim an event is definitely missing.


In [ ]:
def detect_missing_events(
    authoritative_events: list[dict[str, Any]],
    observed_events: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    # Multiset comparison catches missing copies as well as wholly absent IDs.
    source_counts = Counter(event_id(event) for event in authoritative_events)
    observed_counts = Counter(event_id(event) for event in observed_events)
    source_by_id = {event_id(event): event for event in authoritative_events}
    findings = []

    for missing_id, source_count in sorted(source_counts.items()):
        observed_count = observed_counts.get(missing_id, 0)
        if observed_count < source_count:
            source_event = source_by_id[missing_id]
            findings.append({
                "finding_type": "missing_event",
                "severity": "high",
                "event_id": missing_id,
                "expected_occurrences": source_count,
                "observed_occurrences": observed_count,
                "missing_copies": source_count - observed_count,
                "customer_id": str(first_present(source_event, "customer_id", default="unknown")),
                "evidence": {"authoritative_event": source_event},
                "requires_review": True,
            })
    return findings

missing_findings = detect_missing_events(clean_events, observed_events)
print(json.dumps(missing_findings, indent=2, default=str))


## 5. Compare expected and observed invoice amounts

For each invoice line, compare the amount to the corresponding baseline line. The matching key prefers `invoice_line_id`; if your fixture regenerates different line IDs, it falls back to invoice/customer/billing-period/pricing-version identity.

This first version detects invoice amount differences. Later we will also independently recalculate expected amounts from authoritative usage and effective-dated pricing rules.


In [ ]:
def invoice_line_key(line: dict[str, Any]) -> tuple:
    stable_id = first_present(line, "invoice_line_id", "line_id")
    if stable_id is not None:
        return ("id", str(stable_id))
    return (
        "composite",
        str(first_present(line, "invoice_id", default="")),
        str(first_present(line, "customer_id", default="")),
        str(first_present(line, "billing_period", "period", default="")),
        str(first_present(line, "pricing_version_id", default="")),
    )

def compare_expected_and_actual(
    expected_lines: list[dict[str, Any]],
    actual_lines: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    expected_by_key = {invoice_line_key(line): line for line in expected_lines}
    actual_by_key = {invoice_line_key(line): line for line in actual_lines}
    findings = []

    for key in sorted(set(expected_by_key) | set(actual_by_key), key=str):
        expected = expected_by_key.get(key)
        actual = actual_by_key.get(key)

        if expected is None:
            findings.append({
                "finding_type": "unexpected_invoice_line",
                "severity": "medium",
                "line_key": key,
                "expected_amount": "0.00",
                "actual_amount": str(line_amount(actual)),
                "difference": str(line_amount(actual)),
                "requires_review": True,
            })
            continue

        if actual is None:
            findings.append({
                "finding_type": "missing_invoice_line",
                "severity": "high",
                "line_key": key,
                "expected_amount": str(line_amount(expected)),
                "actual_amount": "0.00",
                "difference": str(-line_amount(expected)),
                "requires_review": True,
            })
            continue

        expected_amount = line_amount(expected)
        actual_amount = line_amount(actual)
        difference = money(actual_amount - expected_amount)
        if difference != Decimal("0.00"):
            findings.append({
                "finding_type": "invoice_amount_mismatch",
                "severity": "high" if abs(difference) >= Decimal("5.00") else "medium",
                "line_key": key,
                "customer_id": str(first_present(actual, "customer_id", default=first_present(expected, "customer_id", default="unknown"))),
                "expected_amount": str(expected_amount),
                "actual_amount": str(actual_amount),
                "difference": str(difference),
                "direction": "overcharge" if difference > 0 else "undercharge",
                "evidence": {"expected_line": expected, "actual_line": actual},
                "requires_review": True,
            })
    return findings

invoice_findings = compare_expected_and_actual(baseline_invoice_lines, observed_invoice_lines)
print(json.dumps(invoice_findings, indent=2, default=str))


## 6. Independent calculation verification

This helper verifies invoice line arithmetic using quantity × unit price. It does not trust the recorded `amount` field. Effective-dated pricing selection will be added as a separate deterministic tool in the next refinement.


In [ ]:
def verify_calculation(lines: list[dict[str, Any]]) -> list[dict[str, Any]]:
    findings = []
    for line in lines:
        quantity = decimal_value(first_present(line, "quantity", default=0))
        unit_price = decimal_value(first_present(line, "unit_price", "price_per_unit", default=0))
        recorded_amount = line_amount(line)
        calculated_amount = money(quantity * unit_price)
        if recorded_amount != calculated_amount:
            findings.append({
                "finding_type": "calculation_mismatch",
                "severity": "high",
                "invoice_line_id": str(first_present(line, "invoice_line_id", "line_id", "id", default="unknown")),
                "quantity": str(quantity),
                "unit_price": str(unit_price),
                "recorded_amount": str(recorded_amount),
                "calculated_amount": str(calculated_amount),
                "difference": str(money(recorded_amount - calculated_amount)),
                "requires_review": True,
            })
    return findings

calculation_findings = verify_calculation(observed_invoice_lines)
print(json.dumps(calculation_findings, indent=2, default=str))


## 7. Assemble a reconciliation report

This is the contract the future agents will consume. Findings are grouped by deterministic detector and assigned a case status. This is not an LLM-generated conclusion.


In [ ]:
def run_reconciliation(fixture: dict[str, Any]) -> dict[str, Any]:
    source_events = get_records("clean_events", "source_events", "original_events")
    downstream_events = get_records("observed_events", "processed_events", "ingested_events")
    expected_lines = get_records("baseline_invoice_lines", "correct_invoice_lines", "expected_invoice_lines")
    actual_lines = get_records("observed_invoice_lines", "actual_invoice_lines", "invoice_lines")

    findings = {
        "duplicate_events": detect_duplicate_events(downstream_events),
        "missing_events": detect_missing_events(source_events, downstream_events),
        "invoice_mismatches": compare_expected_and_actual(expected_lines, actual_lines),
        "calculation_mismatches": verify_calculation(actual_lines),
    }
    total = sum(len(items) for items in findings.values())
    return {
        "case_id": "RECON-DEMO-001",
        "status": "review_required" if total else "no_discrepancy_detected",
        "finding_count": total,
        "findings": findings,
        "limitations": [
            "Missing events are provable here only because the fixture contains an authoritative source ledger.",
            "Baseline invoice comparison detects differences but does not yet independently reconstruct every charge from pricing rules.",
            "No invoice or payment system is modified by this notebook.",
        ],
    }

report = run_reconciliation(fixture)
print(json.dumps(report, indent=2, default=str))


## 8. Assertions: fail loudly if the known test cases are not detected

The expected fixture has one duplicate ID, one missing source event, and one undercharged invoice line. If an assertion fails, inspect the schema samples and detector outputs above before continuing.


In [ ]:
assert len(detect_duplicate_events(observed_events)) >= 1, "Expected at least one duplicate event"
assert len(detect_missing_events(clean_events, observed_events)) >= 1, "Expected at least one missing event"
assert any(f["finding_type"] == "invoice_amount_mismatch" for f in invoice_findings), "Expected an invoice amount mismatch"

undercharges = [
    f for f in invoice_findings
    if f.get("finding_type") == "invoice_amount_mismatch" and f.get("direction") == "undercharge"
]
assert undercharges, "Expected an undercharge to be detected"

print("PASS: duplicate-event detector")
print("PASS: missing-event detector")
print("PASS: invoice-mismatch detector")
print("PASS: undercharge classification")
print("All Milestone 2 acceptance checks passed.")


## 9. Export findings for the next milestone

The JSON report is an auditable deterministic output. The next milestone wraps these functions as typed tools for LangChain agents; agents should not reimplement the arithmetic themselves.


In [ ]:
REPORT_PATH = Path("metermind_reconciliation_report.json")
with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print("Report exported to:", REPORT_PATH.resolve())


## Milestone 2 completion checklist

- [ ] Fixture loads from the project directory.
- [ ] Duplicate events are detected.
- [ ] Missing events are detected against the authoritative source ledger.
- [ ] Invoice amount mismatches are detected.
- [ ] Recorded amounts are checked against quantity × unit price.
- [ ] Acceptance assertions pass.
- [ ] `metermind_reconciliation_report.json` is exported.

**Do not add LangGraph agents until these checks pass.**